In [2]:
import sys
from pathlib import Path

import numpy as np

# NumPy 2.x compatibility for fasttext
_original_np_array = np.array

def _fasttext_compatible_array(obj, *args, **kwargs):
    if kwargs.get("copy") is False:
        kwargs["copy"] = None
    return _original_np_array(obj, *args, **kwargs)

np.array = _fasttext_compatible_array

cwd = Path.cwd()
for p in [cwd / 'src', cwd.parent / 'src', cwd / '..' / 'src']:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
        break

from spam_utils import load_split_data
from features import TfidfExtractor, BM25Extractor, FastTextExtractor, BertExtractor
from models import NaiveBayesClassifier, KNNClassifier, FastTextClassifier, FNNClassifier, XGBoostClassifier
from evaluation import evaluate_model, compare_models

import warnings
warnings.filterwarnings('ignore')

In [3]:
# Загрузка готовых сплитов
train_df, test_df = load_split_data()

print(f'Train: {len(train_df)} | Test: {len(test_df)}')
print(f"Train — Ham: {(train_df.label==0).sum()}, Spam: {(train_df.label==1).sum()}")
print(f"Test  — Ham: {(test_df.label==0).sum()}, Spam: {(test_df.label==1).sum()}")
train_df[['label', 'message_clean']].head()

Train: 4120 | Test: 1031
Train — Ham: 3609, Spam: 511
Test  — Ham: 903, Spam: 128


,label,message_clean
0,0,Purity of friendship between two is not about ...
1,0,"Mum, i've sent you many many messages since i ..."
2,0,do u think that any girl will propose u today ...
3,0,Even if he my friend he is a priest call him now
4,0,Thanks honey but still haven't heard anything ...


In [4]:
# TF-IDF + NB
tfidf = TfidfExtractor()
X_train = tfidf.fit_transform(train_df['message_clean'])
X_test  = tfidf.transform(test_df['message_clean'])
y_train = train_df['label']
y_test  = test_df['label']

nb = NaiveBayesClassifier().fit(X_train, y_train)
res_nb = evaluate_model(y_test, nb.predict(X_test), nb.predict_proba(X_test), 'TF-IDF + NB')
res_nb

{'Model': 'TF-IDF + NB',
 'Accuracy': 0.965082444228904,
 'Precision': 1.0,
 'Recall': 0.71875,
 'F1': 0.8363636363636363,
 'ROC-AUC': 0.9771075581395349}

In [5]:
# CBM-25 + KNN
bm25 = BM25Extractor()
X_train = bm25.fit_transform(train_df['message_clean'])
X_test  = bm25.transform(test_df['message_clean'])

knn = KNNClassifier(n_neighbors=5).fit(X_train, y_train)
res_knn = evaluate_model(y_test, knn.predict(X_test), knn.predict_proba(X_test), 'BM-25 + KNN')
res_knn

{'Model': 'BM-25 + KNN',
 'Accuracy': 0.9602327837051406,
 'Precision': 0.9887640449438202,
 'Recall': 0.6875,
 'F1': 0.8110599078341014,
 'ROC-AUC': 0.9269578834440753}

In [ ]:
# FastText
split_dir = Path.cwd().parent / 'data' / 'split' if (Path.cwd().parent / 'data' / 'split').exists() else Path.cwd() / 'data' / 'split'

ft = FastTextClassifier().fit_from_file(str(split_dir / 'fasttext_train.txt'))
res_ft = evaluate_model(
    y_test,
    ft.predict(test_df['message_clean'].tolist()),
    ft.predict_proba(test_df['message_clean'].tolist()),
    'FastText'
)
res_ft

ValueError: invalid literal for int() with base 10: 'ham'

In [13]:
# BERT + FNN
bert = BertExtractor()
X_train_b = bert.fit_transform(train_df['message_clean'].tolist())
X_test_b  = bert.transform(test_df['message_clean'].tolist())

fnn = FNNClassifier().fit(X_train_b, y_train)
res_fnn = evaluate_model(y_test, fnn.predict(X_test_b), fnn.predict_proba(X_test_b), 'BERT + FNN')
res_fnn

Batches: 100%|██████████| 33/33 [00:02<00:00, 13.04it/s]


{'Model': 'BERT + FNN',
 'Accuracy': 0.9786614936954413,
 'Precision': 0.9568965517241379,
 'Recall': 0.8671875,
 'F1': 0.9098360655737705,
 'ROC-AUC': 0.9892199612403101}

In [ ]:
# BERT + XGBoost
xgb = XGBoostClassifier().fit(X_train_b, y_train)
res_xgb = evaluate_model(y_test, xgb.predict(X_test_b), xgb.predict_proba(X_test_b), 'BERT + XGBoost')
res_xgb

{'Model': 'BERT + XGBoost',
 'Accuracy': 0.9796905222437138,
 'Precision': 0.9824561403508771,
 'Recall': 0.8549618320610687,
 'F1': 0.9142857142857143,
 'ROC-AUC': 0.9957478464490714}

In [14]:
# Сводка
compare_models([res_nb, res_knn, res_ft, res_fnn, res_xgb])

NameError: name 'res_ft' is not defined